# UC07 — Segmentación de Riesgo Telemático

Segmentación de conductores basada en datos telemáticos para tarificación dinámica.
**Valor de negocio**: Personalizar primas con factores por perfil de conducción.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS SEGMENTACION_TELEMATICA_DB;
USE SCHEMA SEGMENTACION_TELEMATICA_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos Telemáticos (1.000 conductores, 4 perfiles)

In [ ]:
CREATE OR REPLACE TABLE conductores AS
WITH perfiles AS (SELECT SEQ4() AS id, MOD(SEQ4(),4) AS perfil FROM TABLE(GENERATOR(ROWCOUNT=>1000)))
SELECT 'CON' || LPAD(id,5,'0') AS conductor_id,
    CASE perfil WHEN 0 THEN UNIFORM(40,80,RANDOM()) WHEN 1 THEN UNIFORM(90,140,RANDOM()) WHEN 2 THEN UNIFORM(30,60,RANDOM()) ELSE UNIFORM(50,90,RANDOM()) END::FLOAT AS vel_media,
    CASE perfil WHEN 0 THEN UNIFORM(0,3,RANDOM()) WHEN 1 THEN UNIFORM(8,20,RANDOM()) WHEN 2 THEN UNIFORM(3,8,RANDOM()) ELSE UNIFORM(2,7,RANDOM()) END::FLOAT AS frenazos_100km,
    CASE perfil WHEN 0 THEN UNIFORM(0,2,RANDOM()) WHEN 1 THEN UNIFORM(5,15,RANDOM()) WHEN 2 THEN UNIFORM(2,6,RANDOM()) ELSE UNIFORM(1,5,RANDOM()) END::FLOAT AS aceleraciones_100km,
    CASE perfil WHEN 0 THEN UNIFORM(2,12,RANDOM()) WHEN 1 THEN UNIFORM(10,35,RANDOM()) WHEN 2 THEN UNIFORM(3,15,RANDOM()) ELSE UNIFORM(30,60,RANDOM()) END::FLOAT AS pct_nocturno,
    CASE perfil WHEN 0 THEN UNIFORM(20,50,RANDOM()) WHEN 1 THEN UNIFORM(25,55,RANDOM()) WHEN 2 THEN UNIFORM(60,90,RANDOM()) ELSE UNIFORM(15,45,RANDOM()) END::FLOAT AS pct_urbano,
    CASE perfil WHEN 0 THEN UNIFORM(0,5,RANDOM()) WHEN 1 THEN UNIFORM(12,30,RANDOM()) WHEN 2 THEN UNIFORM(3,10,RANDOM()) ELSE UNIFORM(5,15,RANDOM()) END::FLOAT AS pct_exceso_vel,
    CASE perfil WHEN 0 THEN UNIFORM(75,98,RANDOM()) WHEN 1 THEN UNIFORM(20,50,RANDOM()) WHEN 2 THEN UNIFORM(55,80,RANDOM()) ELSE UNIFORM(45,70,RANDOM()) END::FLOAT AS eco_score,
    UNIFORM(500,3000,RANDOM())::FLOAT AS km_mes, UNIFORM(15,80,RANDOM())::FLOAT AS num_viajes_mes,
    CASE perfil WHEN 0 THEN 'Prudente' WHEN 1 THEN 'Agresivo' WHEN 2 THEN 'Urbano' ELSE 'Nocturno' END AS segmento_real
FROM perfiles;

---

## Paso 3: Train/Test

In [ ]:
CREATE OR REPLACE TABLE entrenamiento AS SELECT * FROM conductores SAMPLE(80);
CREATE OR REPLACE TABLE test AS SELECT * FROM conductores WHERE conductor_id NOT IN (SELECT conductor_id FROM entrenamiento);

---

## Paso 4: Entrenar Clasificador Multi-Clase

In [ ]:
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION segmentador(
    INPUT_DATA=>SYSTEM$REFERENCE('TABLE','entrenamiento'),
    TARGET_COLNAME=>'segmento_real',
    CONFIG_OBJECT=>{'evaluate':TRUE}
);

---

## Paso 5: Evaluar

In [ ]:
CALL segmentador!SHOW_EVALUATION_METRICS();

In [ ]:
CALL segmentador!SHOW_FEATURE_IMPORTANCE();

---

## Paso 6: Puntuar y Asignar Factor de Prima

In [ ]:
CREATE OR REPLACE TABLE predicciones_segmento AS
SELECT conductor_id,
    segmentador!PREDICT(OBJECT_CONSTRUCT('vel_media',vel_media,'frenazos_100km',frenazos_100km,
        'aceleraciones_100km',aceleraciones_100km,'pct_nocturno',pct_nocturno,'pct_urbano',pct_urbano,
        'pct_exceso_vel',pct_exceso_vel,'eco_score',eco_score,'km_mes',km_mes,'num_viajes_mes',num_viajes_mes
    )) AS prediccion,
    prediccion['class']::VARCHAR AS segmento_pred,
    CASE prediccion['class']::VARCHAR WHEN 'Prudente' THEN 0.80 WHEN 'Urbano' THEN 1.00 WHEN 'Nocturno' THEN 1.15 WHEN 'Agresivo' THEN 1.40 ELSE 1.00 END AS factor_prima,
    segmento_real
FROM test;

SELECT segmento_pred, COUNT(*) AS conductores, ROUND(AVG(factor_prima),2) AS factor_medio FROM predicciones_segmento GROUP BY segmento_pred ORDER BY factor_medio;

---

## Paso 7: Dashboard Interactivo

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
st.title('Segmentación de Riesgo Telemático')
df = session.table('predicciones_segmento').to_pandas()

c1,c2,c3 = st.columns(3)
with c1: st.metric('Conductores', len(df))
with c2: st.metric('Exactitud', f"{(df['SEGMENTO_PRED']==df['SEGMENTO_REAL']).mean():.1%}")
with c3: st.metric('Factor Medio', f"{df['FACTOR_PRIMA'].mean():.2f}x")

st.markdown('---')
rc = df['SEGMENTO_PRED'].value_counts()
st.bar_chart(pd.DataFrame({'Conductores': rc.values}, index=rc.index))

st.markdown('---')
st.subheader('Perfil por Segmento')
full = session.table('conductores').to_pandas()
perfil = full.groupby('SEGMENTO_REAL')[['VEL_MEDIA','FRENAZOS_100KM','PCT_NOCTURNO','PCT_URBANO','ECO_SCORE']].mean().round(1)
st.dataframe(perfil, use_container_width=True)

st.markdown('---')
filtro = st.selectbox('Segmento', ['Todos','Prudente','Agresivo','Urbano','Nocturno'])
fdf = df if filtro=='Todos' else df[df['SEGMENTO_PRED']==filtro]
st.dataframe(fdf[['CONDUCTOR_ID','SEGMENTO_PRED','FACTOR_PRIMA','SEGMENTO_REAL']].head(50), use_container_width=True)
st.caption('Desarrollado con Snowflake Cortex ML + Streamlit')

---

## Paso 8: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS SEGMENTACION_TELEMATICA_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;